<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_38_fastapi_sqlalchemy/note_lesson_38_sqlalchemy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗄️ Урок 38 — FastAPI + SQLAlchemy: новинний агрегатор, крок 3

| Крок | Що робимо |
|---|---|
| 0 | завантажуємо `news_hub`, створюємо базу SQLite міграцією Alembic |
| 1 | новини в базі: збір і підрахунок (вправи 1–2) |
| 2 | повний CRUD: 201, 409, PATCH, 204, 404 (вправи 3–6) |
| 3 | SQL через SQLAlchemy: сесія й запити (вправи 7–8) |
| 4 | id і `ON CONFLICT` (вправа 9) |
| 5 | знайди помилку: «перевір, потім встав» (вправа 10) |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам. Рефакторинги, архітектура і вивід на PostgreSQL — у книзі курсу: [Урок 38](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_38/).

---
## 0. Проєкт і база

`news_hub` з уроку 37, де сховище в пам'яті замінила база: `db.py` (engine, сесії, `get_db`), `tables.py` (`NewsRow`), `repository.py` (`NewsRepository`), `migrations/` (Alembic).

У ноутбуці база — файл **SQLite** `lesson38.db`: PostgreSQL не потрібен. Адреса бази — змінна середовища `DATABASE_URL`, її треба задати **до** імпорту `news_hub`. Таблицю створює міграція: `alembic upgrade head`.

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_38_fastapi_sqlalchemy"
REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("news_hub"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import aiosqlite, alembic, fastapi, sqlalchemy  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "news_hub/requirements.txt"], check=True)

os.chdir("news_hub")
sys.path.insert(0, os.getcwd())

os.environ["DATABASE_URL"] = "sqlite+aiosqlite:///lesson38.db"      # до імпорту news_hub!
if os.path.exists("lesson38.db"):
    os.remove("lesson38.db")                                         # щоразу — чиста база
migration = subprocess.run([sys.executable, "-m", "alembic", "upgrade", "head"], capture_output=True, text=True)
print(migration.stderr.strip().splitlines()[-1])

import asyncio

from fastapi.testclient import TestClient
from sqlalchemy import func, select
from sqlalchemy.dialects import postgresql

from news_hub.api import app
from news_hub.db import SessionFactory
from news_hub.models import validate_news
from news_hub.repository import NewsRepository, news_values
from news_hub.snapshot import load_snapshot
from news_hub.tables import NewsRow

client = TestClient(app)
client.__enter__()
print(client.get("/health").json(), "| база:", os.environ["DATABASE_URL"])

---
## 1. Новини в базі

### Вправа 1

Дізнайся поточну версію схеми бази: `alembic current` (через `subprocess.run`, як у клітинці вище). Рядок з результатом — у `current`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
result = subprocess.run([sys.executable, "-m", "alembic", "current"], capture_output=True, text=True)
current = result.stdout.strip()
# END SOLUTION

print(current)
assert current.startswith("0001")
print("✅ Вправа 1 пройдена")

### Вправа 2

Збери знімок (`POST /api/scrape`, `{"source": "snapshot"}`) і збережи кількість новин у базі (`GET /api/news/count`) у `total`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
client.post("/api/scrape", json={"source": "snapshot"})
total = client.get("/api/news/count").json()["count"]
# END SOLUTION

assert total == 168
print("у базі:", total)
print("✅ Вправа 2 пройдена")

**🔮 Прогноз:** Якщо зупинити `TestClient` (як сервер) і створити новий — скільки новин побачить `/api/news/count`? А в уроці 37?

<details>
<summary>Відповідь</summary>

168: новини у файлі `lesson38.db`, а не в пам'яті процесу. В уроці 37 — 0: `NewsStore` створювався в `lifespan` щоразу порожнім.

</details>

In [ ]:
client.__exit__(None, None, None)          # «зупинили сервер»
client = TestClient(app)
client.__enter__()                         # «запустили знову»
print("після перезапуску:", client.get("/api/news/count").json())

---
## 2. Повний CRUD

| Дія | Запит | Успіх | Помилки |
|---|---|---|---|
| створити | `POST /api/news` | `201` | `409` — url уже є, `422` |
| прочитати | `GET /api/news/{id}` | `200` | `404` |
| змінити | `PATCH /api/news/{id}` | `200` | `404`, `422` |
| видалити | `DELETE /api/news/{id}` | `204` | `404` |

### Вправа 3

Створи новину `NEW`. Відповідь (словник) — у `created`.

In [ ]:
NEW = {"title": "НБУ залишив облікову ставку без змін", "url": "https://www.rbc.ua/ukr/news/nbu-rate-778.html",
       "published_time": "14:00"}

# YOUR CODE HERE
# BEGIN SOLUTION
response = client.post("/api/news", json=NEW)
created = response.json()
# END SOLUTION

assert response.status_code == 201
assert (created["lang"], created["category"], created["published_time"]) == ("uk", "Новини", "14:00:00")
print(created)
print("✅ Вправа 3 пройдена")

### Вправа 4

Надішли ту саму новину ще раз. Збережи статус-код у `status_again`. Чому саме такий?

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
status_again = client.post("/api/news", json=NEW).status_code
# END SOLUTION

assert status_again == 409
assert client.get("/api/news/count").json()["count"] == 169
print(status_again, "— url унікальний: це правило таблиці (UNIQUE), а не коду")
print("✅ Вправа 4 пройдена")

### Вправа 5

Зміни категорію створеної новини на `"Економіка"` (`PATCH`), а потім прочитай її **окремим** `GET` — зміна має бути в базі. Результат `GET` — у `reread`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
client.patch(f"/api/news/{created['id']}", json={"category": "Економіка"})
reread = client.get(f"/api/news/{created['id']}").json()
# END SOLUTION

assert reread["category"] == "Економіка" and reread["title"] == NEW["title"]
print(reread["id"], reread["category"], "|", reread["title"])
print("✅ Вправа 5 пройдена")

### Вправа 6

Видали новину. Статус `DELETE` — у `deleted_status`, статус наступного `GET` — у `after_status`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
deleted_status = client.delete(f"/api/news/{created['id']}").status_code
after_status = client.get(f"/api/news/{created['id']}").status_code
# END SOLUTION

assert (deleted_status, after_status) == (204, 404)
print(deleted_status, after_status)
print("✅ Вправа 6 пройдена")

---
## 3. SQL через SQLAlchemy

Сесія — робоче місце: `async with SessionFactory() as session:` відкриває її, `await session.scalar(...)` виконує запит і повертає одне значення, `await session.scalars(...)` — список об'єктів. У Jupyter `await` можна писати прямо в клітинці.

### Вправа 7

Порахуй новини російською **запитом до бази**: `select(func.count()).select_from(NewsRow).where(...)`. Результат — у `ru_count`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
async with SessionFactory() as session:
    ru_count = await session.scalar(select(func.count()).select_from(NewsRow).where(NewsRow.lang == "ru"))
# END SOLUTION

assert ru_count == client.get("/api/news/stats").json()["lang"]["ru"] == 138
print("ru:", ru_count)
print("✅ Вправа 7 пройдена")

### Вправа 8

Склади запит `longest`: три новини з найдовшими заголовками (`func.length(NewsRow.title)`, спадання). Виконай його і подивись, який SQL він дає для PostgreSQL.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
longest = select(NewsRow).order_by(func.length(NewsRow.title).desc()).limit(3)
# END SOLUTION

async with SessionFactory() as session:
    rows = list(await session.scalars(longest))
print(longest.compile(dialect=postgresql.dialect()), "\n")
for row in rows:
    print(len(row.title), row.title[:70])
assert len(rows) == 3 and len(rows[0].title) >= len(rows[1].title) >= len(rows[2].title)
print("✅ Вправа 8 пройдена")

---
## 4. id і `ON CONFLICT`

`add_many` робить `INSERT … ON CONFLICT (url) DO NOTHING`: дублікати відсіює база.

**🔮 Прогноз:** Зберемо знімок ще раз (усі 168 url уже є), а потім створимо нову новину. Який id вона отримає в SQLite? А в PostgreSQL (сторінка уроку)?

<details>
<summary>Відповідь</summary>

SQLite: **169**. `INTEGER PRIMARY KEY` бере найбільший наявний id + 1: зараз найбільший — 168 (новину 169 ми видалили у вправі 6), тож номер 169 видається **повторно**, а пропущені при повторному зборі рядки номерів не забирають. PostgreSQL бере значення з послідовності для **кожного** рядка, навіть пропущеного, і ніколи не повторює: на сторінці уроку нова новина отримала id 337.

</details>

### Вправа 9

Збери знімок удруге (звіт — у `again`), створи новину `OTHER` і збережи її id у `other_id`. Порівняй зі своїм прогнозом.

In [ ]:
OTHER = {"title": "Гривня зміцнилася до долара на міжбанку", "url": "https://www.rbc.ua/ukr/news/hryvnia-777.html"}

# YOUR CODE HERE
# BEGIN SOLUTION
again = client.post("/api/scrape", json={"source": "snapshot"}).json()
other_id = client.post("/api/news", json=OTHER).json()["id"]
# END SOLUTION

assert again["news_saved"] == 0
print("нових при повторному зборі:", again["news_saved"], "| id нової новини:", other_id)
assert other_id < 200, "у SQLite пропущені рядки номерів не забирають"
print("✅ Вправа 9 пройдена")

---
## 5. Знайди помилку

«Спрощений» збір: для кожної новини спершу перевірити, чи є url, і лише тоді додати.

```python
exists = await session.scalar(select(NewsRow.id).where(NewsRow.url == url))
if exists is None:
    session.add(NewsRow(...))
```

З одним користувачем — працює. Запустимо **два збори одночасно** (`asyncio.gather`).

In [ ]:
fresh, _ = validate_news(load_snapshot()[:20])
fresh = [item.model_copy(update={"url": str(item.url).replace(".html", "-race.html")}) for item in fresh]   # нові url


async def naive_add_many(items):
    async with SessionFactory() as session:
        saved = 0
        for item in items:
            exists = await session.scalar(select(NewsRow.id).where(NewsRow.url == str(item.url)))
            if exists is None:
                session.add(NewsRow(**news_values(item)))
                saved += 1
        await session.commit()
        return saved


results = await asyncio.gather(naive_add_many(fresh), naive_add_many(fresh), return_exceptions=True)
for result in results:
    print(type(result).__name__, str(result).splitlines()[0][:100])

### Вправа 10

Перепиши збір правильно: `fixed_add_many(items)` через `NewsRepository(session).add_many(items)` (там `INSERT … ON CONFLICT DO NOTHING`) і `commit()`. Два одночасні виклики не повинні падати, а разом — додати кожну новину рівно раз.

In [ ]:
fresh2 = [item.model_copy(update={"url": str(item.url).replace("-race.html", "-fixed.html")}) for item in fresh]


# YOUR CODE HERE
# BEGIN SOLUTION
async def fixed_add_many(items):
    async with SessionFactory() as session:
        saved = await NewsRepository(session).add_many(items)
        await session.commit()
        return saved
# END SOLUTION

results = await asyncio.gather(fixed_add_many(fresh2), fixed_add_many(fresh2), return_exceptions=True)
print(results)
assert not any(isinstance(result, Exception) for result in results) and sum(results) == 20
print("✅ Вправа 10 пройдена")

---
## Самоперевірка

1. Чим `NewsRow` відрізняється від `NewsItem`?
2. Що робить `get_db` і навіщо `scope="function"`?
3. Чому `POST` тієї самої новини — `409`, а не `500`?
4. Навіщо міграції, якщо є `Base.metadata.create_all()`?
5. Чому «перевір, потім встав» ламається під паралельними запитами?

<details>
<summary>Відповіді</summary>

1. `NewsItem` (Pydantic) перевіряє дані з парсера й запиту; `NewsRow` (SQLAlchemy) — рядок таблиці з `id`, `UNIQUE (url)`, `scraped_at`.
2. Дає кожному запиту сесію й транзакцію: без винятку — COMMIT, з винятком — ROLLBACK. `scope="function"` — щоб COMMIT був до відповіді: інакше (FastAPI ≥ 0.118) клієнт отримав би успіх навіть при невдалому COMMIT.
3. База відхиляє дублікат (`UNIQUE`), `repo.create` робить `flush()` — помилка виникає в ендпоінті, і він перетворює `IntegrityError` на `409`.
4. `create_all` лише створює відсутні таблиці; міграції змінюють наявну схему без втрати даних, по версіях, з відкатом.
5. Між `SELECT` і `COMMIT` інший запит встигає зробити те саме; правило унікальності треба віддати базі — `ON CONFLICT DO NOTHING` / `UNIQUE`.

</details>

## Далі

- Книга курсу: [Урок 38](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_38/) — SQL, який генерує SQLAlchemy, COMMIT до відповіді, міграції, архітектура, вивід на PostgreSQL.
- У папці `news_hub`: `docker compose up -d db`, `alembic upgrade head`, `uvicorn news_hub.api:app --reload`; `pytest`.
- **Урок 39** — middleware і Redis: кеш і rate limit.

In [ ]:
client.__exit__(None, None, None)